In [1]:
from pathlib import Path

ROOT = Path("/kaggle/input/datasets/riyasuryawanshi/scan-vs-span")

for name in [
    "train_phase3.py",
    "scan_vs_span.py",
    "dataset.py",
    "model.py",
    "metrics.py",
    "CUAD_v1.json",
    "clauses_mapped.jsonl",
    "partial/config.json",
    "partial/splits.json",
]:
    p = ROOT / name
    print(("OK" if p.exists() else "MISSING"), name)

text = (ROOT / "train_phase3.py").read_text()
print("\ncandidate_spans exists:", "def candidate_spans(" in text)

OK train_phase3.py
OK scan_vs_span.py
OK dataset.py
OK model.py
OK metrics.py
OK CUAD_v1.json
OK clauses_mapped.jsonl
OK partial/config.json
OK partial/splits.json

candidate_spans exists: True


In [2]:

from pathlib import Path
import shutil
import os

ROOT = Path("/kaggle/input/datasets/riyasuryawanshi/scan-vs-span")
WORK = Path("/kaggle/working/cuad_project")

shutil.copytree(ROOT, WORK, dirs_exist_ok=True)
os.chdir(WORK)

print("Project copied successfully.")
print("Working directory:", os.getcwd())


Project copied successfully.
Working directory: /kaggle/working/cuad_project


In [3]:

!python -B /kaggle/working/cuad_project/scan_vs_span.py \
  --run_dir /kaggle/working/cuad_project \
  --fold 0 \
  --ckpt_span /kaggle/working/cuad_project/ckpt_joint_cls_span_fold0.pt \
  --ckpt_cls /kaggle/working/cuad_project/ckpt_cls_only_fold0.pt \
  --mapped_jsonl /kaggle/working/cuad_project/clauses_mapped.jsonl \
  --cuad_json /kaggle/working/cuad_project/CUAD_v1.json \
  --n_windows 20 \
  --n_boot 100


fold 0: train 6914, test 1983
config.json: 100%|█████████████████████████████| 694/694 [00:00<00:00, 3.37MB/s]
vocab.json: 100%|████████████████████████████| 899k/899k [00:00<00:00, 25.3MB/s]
merges.txt: 100%|█████████████████████████████| 456k/456k [00:00<00:00, 105MB/s]
tokenizer.json: 100%|██████████████████████| 1.36M/1.36M [00:00<00:00, 57.8MB/s]
20 windows

pytorch_model.bin: downloading bytes: █████▊                |  156MB, 12.5MB/s  
pytorch_model.bin: downloading bytes: ████████▌             |  233MB, 19.6MB/s  
pytorch_model.bin: downloading bytes: █████████▉            |  269MB, 21.8MB/s  
pytorch_model.bin: downloading bytes: ███████████▎          |  308MB, 27.1MB/s  
pytorch_model.bin: downloading bytes: █████████████         |  354MB, 28.0MB/s  
pytorch_model.bin: downloading bytes: ██████████████████████|  363MB, 29.1MB/s  
pytorch_model.bin: reconstructing file: 100%|██████|  597MB /  597MB, 49.3MB/s  

model.safetensors: downloading bytes: ███████████▍          |  310

In [4]:

%cd /kaggle/working/cuad_project

!python -B scan_vs_span.py \
  --run_dir /kaggle/working/cuad_project \
  --fold 0 \
  --ckpt_span /kaggle/working/cuad_project/ckpt_joint_cls_span_fold0.pt \
  --ckpt_cls /kaggle/working/cuad_project/ckpt_cls_only_fold0.pt \
  --mapped_jsonl /kaggle/working/cuad_project/clauses_mapped.jsonl \
  --cuad_json /kaggle/working/cuad_project/CUAD_v1.json \
  --n_windows 500 \
  --n_boot 2000


/kaggle/working
fold 0: train 6914, test 1983
500 windows
Loading weights: 100%|██████████████████████| 269/269 [00:00<00:00, 9587.43it/s]
[transformers] LongformerModel LOAD REPORT from: allenai/longformer-base-4096
Key                            | Status     |  | 
-------------------------------+------------+--+-
lm_head.dense.weight           | UNEXPECTED |  | 
lm_head.dense.bias             | UNEXPECTED |  | 
lm_head.layer_norm.bias        | UNEXPECTED |  | 
lm_head.decoder.weight         | UNEXPECTED |  | 
longformer.pooler.dense.bias   | UNEXPECTED |  | 
lm_head.bias                   | UNEXPECTED |  | 
longformer.pooler.dense.weight | UNEXPECTED |  | 
lm_head.layer_norm.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████████████████| 269/269 [00:00<00:00, 7301.14it/s]
[transformers] LongformerModel LOAD REPORT from: allenai/longformer-base-4096
Key

In [5]:

from IPython.display import display, Markdown
from pathlib import Path

report = Path(
    "/kaggle/working/cuad_project/scan_vs_span_fold0.md"
)

assert report.exists(), "Report not found. Check whether evaluation completed."

display(Markdown(report.read_text()))
print("\nReport saved at:", report)


# Span head vs classifier scan (fold 0, 500 windows)

| Method | forward passes / window | ms / window | token-F1 | exact match | IoU |
|---|---|---|---|---|---|
| span head | 1 | 54 | 0.748 | 0.494 | 0.698 |
| classifier scan | 50 | 1303 | 0.615 | 0.134 | 0.556 |

Candidate ceiling (best any candidate could score): token-F1 0.972.
Span head minus scan, token-F1: +0.132 (95% bootstrap interval +0.092 to +0.170).

One fold, a random sample of test windows, windowed task. Speed is wall-clock on this hardware.


Report saved at: /kaggle/working/cuad_project/scan_vs_span_fold0.md


In [6]:

from IPython.display import FileLink, display

display(FileLink(
    "/kaggle/working/cuad_project/scan_vs_span_fold0.md"
))


/kaggle/working/cuad_project/scan_vs_span_fold0.md

In [23]:

from pathlib import Path
import shutil
import os

ROOT = Path("/kaggle/input/datasets/riyasuryawanshi/scan-vs-span")
WORK = Path("/kaggle/working/cuad_project")

assert ROOT.exists(), f"Dataset not found: {ROOT}"

# Copy updated project files and checkpoints
shutil.copytree(ROOT, WORK, dirs_exist_ok=True)

os.chdir(WORK)

print("Working directory updated:", WORK)


Working directory updated: /kaggle/working/cuad_project


In [24]:

from pathlib import Path

for p in Path("/kaggle/input").rglob("ckpt_cls_only_fold1.pt"):
    print("Found:", p)

for p in Path("/kaggle/input").rglob("ckpt_joint_cls_span_fold1.pt"):
    print("Found:", p)

for p in Path("/kaggle/input").rglob("ckpt_cls_only_fold2.pt"):
    print("Found:", p)

for p in Path("/kaggle/input").rglob("ckpt_joint_cls_span_fold2.pt"):
    print("Found:", p)


Found: /kaggle/input/datasets/riyasuryawanshi/scan-vs-span/ckpt_cls_only_fold1.pt
Found: /kaggle/input/datasets/riyasuryawanshi/scan-vs-span/ckpt_joint_cls_span_fold1.pt
Found: /kaggle/input/datasets/riyasuryawanshi/scan-vs-span/ckpt_cls_only_fold2.pt
Found: /kaggle/input/datasets/riyasuryawanshi/scan-vs-span/ckpt_joint_cls_span_fold2.pt


In [25]:

from pathlib import Path
import shutil

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working/cuad_project")

required = [
    "ckpt_cls_only_fold1.pt",
    "ckpt_joint_cls_span_fold1.pt",
    "ckpt_cls_only_fold2.pt",
    "ckpt_joint_cls_span_fold2.pt",
]

for name in required:
    matches = list(INPUT.rglob(name))
    if not matches:
        print("MISSING FROM INPUT:", name)
        continue

    source = matches[0]
    destination = WORK / name
    shutil.copy2(source, destination)
    print("Copied:", name, "from", source)


Copied: ckpt_cls_only_fold1.pt from /kaggle/input/datasets/riyasuryawanshi/scan-vs-span/ckpt_cls_only_fold1.pt
Copied: ckpt_joint_cls_span_fold1.pt from /kaggle/input/datasets/riyasuryawanshi/scan-vs-span/ckpt_joint_cls_span_fold1.pt
Copied: ckpt_cls_only_fold2.pt from /kaggle/input/datasets/riyasuryawanshi/scan-vs-span/ckpt_cls_only_fold2.pt
Copied: ckpt_joint_cls_span_fold2.pt from /kaggle/input/datasets/riyasuryawanshi/scan-vs-span/ckpt_joint_cls_span_fold2.pt


In [26]:

for fold in [0, 1, 2]:
    for prefix in ["ckpt_cls_only", "ckpt_joint_cls_span"]:
        p = WORK / f"{prefix}_fold{fold}.pt"
        print(("OK" if p.exists() else "MISSING"), p.name)


OK ckpt_cls_only_fold0.pt
OK ckpt_joint_cls_span_fold0.pt
OK ckpt_cls_only_fold1.pt
OK ckpt_joint_cls_span_fold1.pt
OK ckpt_cls_only_fold2.pt
OK ckpt_joint_cls_span_fold2.pt


In [27]:

!python -B scan_vs_span.py \
  --run_dir /kaggle/working/cuad_project \
  --fold 1 \
  --ckpt_span /kaggle/working/cuad_project/ckpt_joint_cls_span_fold1.pt \
  --ckpt_cls /kaggle/working/cuad_project/ckpt_cls_only_fold1.pt \
  --mapped_jsonl /kaggle/working/cuad_project/clauses_mapped.jsonl \
  --cuad_json /kaggle/working/cuad_project/CUAD_v1.json \
  --n_windows 500 \
  --n_boot 2000


fold 1: train 6658, test 2043
500 windows
Loading weights: 100%|██████████████████████| 269/269 [00:00<00:00, 7802.30it/s]
[transformers] LongformerModel LOAD REPORT from: allenai/longformer-base-4096
Key                            | Status     |  | 
-------------------------------+------------+--+-
lm_head.dense.bias             | UNEXPECTED |  | 
lm_head.bias                   | UNEXPECTED |  | 
longformer.pooler.dense.weight | UNEXPECTED |  | 
longformer.pooler.dense.bias   | UNEXPECTED |  | 
lm_head.layer_norm.bias        | UNEXPECTED |  | 
lm_head.decoder.weight         | UNEXPECTED |  | 
lm_head.layer_norm.weight      | UNEXPECTED |  | 
lm_head.dense.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████████████████| 269/269 [00:00<00:00, 8232.59it/s]
[transformers] LongformerModel LOAD REPORT from: allenai/longformer-base-4096
Key                

In [28]:
from IPython.display import display, Markdown
from pathlib import Path

report = Path("/kaggle/working/cuad_project/scan_vs_span_fold1.md")
display(Markdown(report.read_text()))

# Span head vs classifier scan (fold 1, 500 windows)

| Method | forward passes / window | ms / window | token-F1 | exact match | IoU |
|---|---|---|---|---|---|
| span head | 1 | 54 | 0.755 | 0.510 | 0.703 |
| classifier scan | 50 | 1300 | 0.613 | 0.122 | 0.552 |

Candidate ceiling (best any candidate could score): token-F1 0.959.
Span head minus scan, token-F1: +0.142 (95% bootstrap interval +0.105 to +0.180).

One fold, a random sample of test windows, windowed task. Speed is wall-clock on this hardware.

In [29]:

!python -B scan_vs_span.py \
  --run_dir /kaggle/working/cuad_project \
  --fold 2 \
  --ckpt_span /kaggle/working/cuad_project/ckpt_joint_cls_span_fold2.pt \
  --ckpt_cls /kaggle/working/cuad_project/ckpt_cls_only_fold2.pt \
  --mapped_jsonl /kaggle/working/cuad_project/clauses_mapped.jsonl \
  --cuad_json /kaggle/working/cuad_project/CUAD_v1.json \
  --n_windows 500 \
  --n_boot 2000


fold 2: train 6569, test 2070
500 windows
Loading weights: 100%|█████████████████████| 269/269 [00:00<00:00, 12945.98it/s]
[transformers] LongformerModel LOAD REPORT from: allenai/longformer-base-4096
Key                            | Status     |  | 
-------------------------------+------------+--+-
lm_head.dense.bias             | UNEXPECTED |  | 
lm_head.layer_norm.weight      | UNEXPECTED |  | 
lm_head.dense.weight           | UNEXPECTED |  | 
lm_head.decoder.weight         | UNEXPECTED |  | 
longformer.pooler.dense.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias        | UNEXPECTED |  | 
lm_head.bias                   | UNEXPECTED |  | 
longformer.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████████████████| 269/269 [00:00<00:00, 9002.74it/s]
[transformers] LongformerModel LOAD REPORT from: allenai/longformer-base-4096
Key                

In [30]:
from IPython.display import display, Markdown
from pathlib import Path

report = Path("/kaggle/working/cuad_project/scan_vs_span_fold2.md")
display(Markdown(report.read_text()))

# Span head vs classifier scan (fold 2, 500 windows)

| Method | forward passes / window | ms / window | token-F1 | exact match | IoU |
|---|---|---|---|---|---|
| span head | 1 | 54 | 0.764 | 0.540 | 0.713 |
| classifier scan | 51 | 1340 | 0.662 | 0.106 | 0.592 |

Candidate ceiling (best any candidate could score): token-F1 0.963.
Span head minus scan, token-F1: +0.102 (95% bootstrap interval +0.066 to +0.139).

One fold, a random sample of test windows, windowed task. Speed is wall-clock on this hardware.